# GridCast: fair Ridge vs LightGBM resource pilot (Colab, CPU only)

**Step 1: introduction and experiment boundaries.**

This notebook prepares and, only after an explicit opt-in, runs the
**six-fit real-data resource pilot** from
`docs/FAIR_MODEL_COMPARISON_RESOURCE_PILOT.md` on a free Colab **CPU** runtime.

| Candidate | Horizons |
|---|---|
| `r01` Ridge, alpha 1.0 | 1 h, 168 h |
| `l02` LightGBM L2, 600 trees, 63 leaves | 1 h, 168 h |
| `l11` LightGBM L1, 300 trees, 63 leaves | 1 h, 168 h |

The pilot **measures resources only** (fit/prediction time, peak memory, model size).
It produces no MAE table, no selection and no winner.

What this notebook will **never** do:

* run the 204-fit historical search or the 32 primary-fold refits;
* evaluate primary-test folds;
* change the grid, training period, features, seeds, thread limits or caps;
* use a GPU/TPU, paid resources or background jobs;
* upload raw data or embed credentials or private paths.

**Default behaviour is preparation and inspection only.** Running every cell top to
bottom with defaults performs no model fit. The pilot runs only if you set
`AUTHORIZE_SIX_FIT_PILOT = True` in Step 8, which requires separate approval.

Verdicts: **READY**: all hard checks pass. **CONDITIONAL**: hard checks pass with
recorded caveats (e.g. pressure-stall data unavailable). **BLOCKED**: a hard check
failed; do not run the pilot. READY is not authorization.

Guide: `docs/COLAB_FAIR_COMPARISON_GUIDE.md`.

## Step 2: runtime information and CPU-only verification
Select **Runtime → Change runtime type → CPU** first. This cell uses only the standard library.

In [ ]:
# STEP 2: runtime information and CPU-only verification
import os, platform, sys
from pathlib import Path

print("Python (notebook kernel):", sys.version.split()[0], "|", platform.platform())
print("Colab release:", os.environ.get("COLAB_RELEASE_TAG", "not detected"))
accelerators = [p for p in ("/dev/nvidia0", "/dev/nvidiactl", "/dev/accel0") if Path(p).exists()]
if os.environ.get("COLAB_GPU", "") not in ("", "0"):
    accelerators.append("COLAB_GPU")
accelerators += [k for k in ("COLAB_TPU_ADDR", "TPU_NAME") if os.environ.get(k)]
if accelerators:
    raise SystemExit(f"BLOCKED: accelerator attached {accelerators}. Switch to a CPU runtime and restart.")
if platform.system() != "Linux":
    raise SystemExit("BLOCKED: this notebook targets the Linux Colab runtime.")
meminfo = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
print("CPUs (logical):", os.cpu_count(), "| affinity:", len(os.sched_getaffinity(0)))
print("MemTotal:", meminfo["MemTotal"].strip(), "| MemAvailable:", meminfo["MemAvailable"].strip())
for f in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    if Path(f).exists():
        print("cgroup memory limit:", f, "=", Path(f).read_text().strip())
print("CPU-only runtime confirmed.")

## Step 3: repository checkout at a pinned commit
The GitHub repository is **private**. Choose one source:

* `bundle` (recommended): on your own machine run
  `git bundle create gridcast.bundle main`, upload `gridcast.bundle` to a Drive folder
  of your choice, set `MOUNT_DRIVE = True` and paste its path into `REPO_BUNDLE_PATH`.
* `github`: add a read-only fine-grained token as a **Colab Secret** named
  `GITHUB_TOKEN` (key icon in the left sidebar). It is read at run time, never printed
  or written to disk. If the repository is public, no secret is needed.

The processed EIA-930 snapshot (15 MB) is tracked in the repository, so the checkout
also provides the dataset; Step 5 verifies its fingerprint.

In [ ]:
# STEP 3: repository checkout at a pinned commit
import base64, subprocess

PINNED_COMMIT = "32be70b38d2f75745d4fb69a7cc8e4083ec08d3b"  #@param {type:"string"}
REPO_SOURCE = "bundle"  #@param ["bundle", "github"]
REPO_BUNDLE_PATH = ""  #@param {type:"string"}
MOUNT_DRIVE = False  #@param {type:"boolean"}
GITHUB_REPOSITORY = "AdebanjiAdelowo/electricity-demand-forecasting"
WORKDIR = Path("/content/gridcast")

def sh(cmd, **kw):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, text=True, **kw)

if len(PINNED_COMMIT) != 40:
    raise SystemExit("Set PINNED_COMMIT to the full 40-character commit SHA to run.")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if not WORKDIR.exists():
    if REPO_SOURCE == "bundle":
        if not REPO_BUNDLE_PATH or not Path(REPO_BUNDLE_PATH).is_file():
            raise SystemExit("Set REPO_BUNDLE_PATH to your uploaded gridcast.bundle (see the guide).")
        sh(["git", "clone", "--no-checkout", REPO_BUNDLE_PATH, WORKDIR])
    else:
        env = dict(os.environ)
        try:
            from google.colab import userdata
            secret = userdata.get("GITHUB_TOKEN")
        except Exception:
            secret = None
        if secret:
            basic = base64.b64encode(f"x-access-token:{secret}".encode()).decode()
            env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
        sh(["git", "clone", "--no-checkout", f"https://github.com/{GITHUB_REPOSITORY}.git", WORKDIR], env=env)
        del secret, env
    sh(["git", "-C", WORKDIR, "checkout", "--quiet", "--detach", PINNED_COMMIT])
head = subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "-C", str(WORKDIR), "status", "--porcelain"], text=True).strip()
if head != PINNED_COMMIT or dirty:
    raise SystemExit(f"BLOCKED: checkout {head} (dirty={bool(dirty)}) does not match pinned {PINNED_COMMIT}. "
                     "Use a fresh runtime; nothing is altered or resumed silently.")
print("Checked out", head, "at", WORKDIR)

## Step 4: dependency installation from the locked environment
Installs a pinned `uv` into the disposable Colab VM, then creates the project's own
`.venv` from `uv.lock` with `--frozen` (no resolution, no upgrades, **core dependencies
only**: no torch, CUDA or GPU packages). If Colab's Python is not 3.11–3.12, uv fetches
a managed Python 3.12 (~30 MB). Expect roughly 0.5–1 GB of disk and 1–3 minutes.

In [ ]:
# STEP 4: dependency installation from the locked environment
import json
UV_VERSION = "0.11.16"
sh([sys.executable, "-m", "pip", "install", "--quiet", f"uv=={UV_VERSION}"])
sh(["uv", "sync", "--frozen", "--python", "3.12"], cwd=WORKDIR)
VENV_PY = WORKDIR / ".venv/bin/python"
THREADS = "2"
CHILD_ENV = {**os.environ, "GRIDCAST_ROOT": str(WORKDIR), "CUDA_VISIBLE_DEVICES": "",
             **{k: THREADS for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                                      "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "GRIDCAST_JOBS")}}

def gridcast(*args, check=False):
    cmd = [str(VENV_PY), "-m", "gridcast.experiments.colab_pilot", *map(str, args)]
    print("+", " ".join(cmd[2:]))
    return subprocess.run(cmd, cwd=WORKDIR, env=CHILD_ENV, text=True, capture_output=True, check=check)

def parse_result(stdout):
    lines = [ln for ln in stdout.splitlines() if ln.startswith("GRIDCAST_RESULT ")]
    return json.loads(lines[-1].split(" ", 1)[1]) if lines else {}

sh([VENV_PY, "-c", "import lightgbm, sklearn, pandas, numpy; print('lightgbm', lightgbm.__version__, "
    "'sklearn', sklearn.__version__, 'pandas', pandas.__version__, 'numpy', numpy.__version__)"], cwd=WORKDIR)

## Step 5: dataset availability and fingerprint verification
The checkout already contains `data/processed/ercot_hourly.parquet` and `manifest.json`.
Only if they are missing (for example an archive without data), set
`DATASET_SOURCE_DIR` to a folder you provide containing exactly those two files; they are
linked, not copied, and only after matching the approved SHA-256. Verification checks
SHA-256, schema, UTC index, demand/imputation columns, coverage and provenance, and
**fails closed** on any difference.

In [ ]:
# STEP 5: dataset availability and fingerprint verification
DATASET_SOURCE_DIR = ""  #@param {type:"string"}
if DATASET_SOURCE_DIR:
    staged = gridcast("stage-dataset", DATASET_SOURCE_DIR)
    print(staged.stdout, staged.stderr)
    if staged.returncode:
        raise SystemExit("BLOCKED: provided dataset does not match the approved snapshot.")
check = subprocess.run([str(VENV_PY), "-c", "import json; from gridcast.data import snapshot; "
                        "print(json.dumps(snapshot.verify(), default=str))"],
                       cwd=WORKDIR, env=CHILD_ENV, text=True, capture_output=True)
if check.returncode:
    raise SystemExit(check.stderr)
DATASET = json.loads(check.stdout)
print(json.dumps({k: DATASET.get(k) for k in ("ok", "dataset_version", "processed_sha256", "problems")}, indent=2))
if not DATASET["ok"]:
    raise SystemExit("BLOCKED: approved EIA-930 snapshot unavailable or altered. See the guide; no training.")

## Step 6: Linux memory and disk preflight
Reads `/proc/meminfo`, the container cgroup memory limit/usage, pressure-stall data,
process RSS, disk, CPU affinity/quota and the lockfile versions. **No data values are
decoded and no model is built.** Admission requires effective available memory ≥
(2,048 MiB ceiling − current RSS) + 1,024 MiB margin; a large host never overrides a
smaller container limit, and an unreadable container limit is BLOCKED.

In [ ]:
# STEP 6: Linux memory and disk preflight
result = gridcast("preflight", "--expected-commit", PINNED_COMMIT)
print(result.stdout or result.stderr)
summary = parse_result(result.stdout) or {"verdict": "BLOCKED", "blocking": ["preflight did not complete"]}
PREFLIGHT_DIR = Path(summary["output"]) if "output" in summary else None
PREFLIGHT = json.loads((PREFLIGHT_DIR / "preflight.json").read_text()) if PREFLIGHT_DIR else {}
res = PREFLIGHT.get("resources", {})
mib = lambda b: None if b is None else round(b / 2**20)
print("Runtime:", PREFLIGHT.get("runtime"), "| CPU:", res.get("cpu"))
print("MemAvailable MiB:", mib(res.get("mem_available_bytes")), "| container available MiB:",
      mib(res.get("container_available_bytes")), "| effective MiB:", mib(res.get("effective_available_bytes")))
print("cgroup:", res.get("cgroup"), "| disk free GiB:", round((res.get("disk_free_bytes") or 0) / 2**30, 1))
print("VERDICT:", summary["verdict"])

## Step 7: dry-run experiment configuration inspection
Prints the fixed pilot plan from the unchanged reviewed configuration. No fit.

In [ ]:
# STEP 7: dry-run experiment configuration inspection
plan = PREFLIGHT.get("pilot_plan") or json.loads(gridcast(check=True).stdout)
for c in plan["candidates"]:
    print(c["id"], c["model"], json.dumps(c["parameters"], sort_keys=True))
print("horizons:", plan["horizons"], "| fit start:", plan["fit_start"], "| target cutoff:", plan["target_cutoff"],
      "| validation end:", plan["validation_end"])
print("planned fits:", plan["planned_fits"], "| wall cap s:", plan["seconds_cap"],
      "| memory ceiling MiB:", plan["nominal_memory_mib"], "| early stop MiB:", plan["early_stop_mib"],
      "| threads:", plan["threads"])
assert plan["planned_fits"] == 6 and plan["config_unchanged"]

## Step 8: explicitly gated six-fit resource pilot
**Do not enable without separate written approval.** When enabled, a fresh preflight
runs first; if BLOCKED nothing is fitted. The pilot runs in the foreground in a
supervised child process (2 threads, no CUDA). The supervisor kills it if process-tree
RSS exceeds 2,048 MiB or wall time exceeds 330 s; the pilot's own guards stop at
1,843 MiB / 300 s. A new append-only directory is created; nothing is resumed.

In [ ]:
# STEP 8: explicitly gated six-fit resource pilot
AUTHORIZE_SIX_FIT_PILOT = False  #@param {type:"boolean"}
PILOT_DIR = None
if not AUTHORIZE_SIX_FIT_PILOT:
    print("Pilot NOT run (default). Preparation and inspection only.")
elif summary.get("verdict") == "BLOCKED":
    raise SystemExit("Preflight is BLOCKED; the pilot is not admitted.")
else:
    run = gridcast("pilot", "--authorize-six-fit-pilot", "--expected-commit", PINNED_COMMIT)
    print(run.stdout, run.stderr[-4000:])
    outcome = parse_result(run.stdout)
    PILOT_DIR = Path(outcome["output"]) if "output" in outcome else None
    print("Pilot outcome:", outcome.get("label", "FAILED BEFORE LAUNCH"))

## Step 9: resource-measurement inspection
Shows each fit's training rows, feature count, fit/prediction time, process peak RSS and
serialized model size, plus any interrupted or failed fit. Earlier pilot directories on
this VM are listed too; incomplete runs are never reported as a completed pilot.

In [ ]:
# STEP 9: resource-measurement inspection
import pandas as pd

results_root = WORKDIR / "results/fair_model_comparison"
for directory in sorted(results_root.glob("resource_pilot_colab_*")):
    report = json.loads(gridcast("inspect", directory, check=True).stdout)
    print(directory.name, "->", report["label"], "| measured:", len(report["measured_fits"]),
          "| incomplete:", report["incomplete_fits"], "| problems:", report["problems"])
    if directory == PILOT_DIR and report["measurements"]:
        table = pd.DataFrame(report["measurements"])
        table["peak_rss_mib"] = (table.pop("peak_rss_bytes") / 2**20).round(1)
        display(table)
if PILOT_DIR is None:
    print("No pilot run in this session; nothing measured.")

## Step 10: export of results and provenance
Set `EXPORT_DIR` to an existing persistent folder you choose (e.g. inside mounted Drive).
Each export is a new zip with run metadata, configuration, measurements, fit ledger,
supervisor record and SHA-256 manifest. **Fitted models and prediction tables (which
contain observed demand) are excluded.** Existing archives are never overwritten.

In [ ]:
# STEP 10: export of results and provenance
EXPORT_DIR = ""  #@param {type:"string"}
EXPORTS = []
if not EXPORT_DIR:
    print("EXPORT_DIR not set; nothing exported. Colab VM files are lost when the runtime ends.")
else:
    for directory in [d for d in (PREFLIGHT_DIR, PILOT_DIR) if d]:
        exported = gridcast("export", directory, "--destination", EXPORT_DIR)
        print(exported.stdout or exported.stderr)
        if exported.returncode == 0:
            EXPORTS.append(exported.stdout.strip())

## Step 11: final readiness report

In [ ]:
# STEP 11: final readiness report
pilot_label = "NOT RUN (not authorized in this session)"
if PILOT_DIR:
    pilot_label = json.loads(gridcast("inspect", PILOT_DIR, check=True).stdout)["label"]
report = {
    "pinned_commit": PINNED_COMMIT,
    "dataset_verified": DATASET.get("ok"),
    "preflight_verdict": summary.get("verdict"),
    "preflight_blocking": summary.get("blocking"),
    "preflight_warnings": summary.get("warnings"),
    "six_fit_pilot": pilot_label,
    "exports": EXPORTS,
    "full_historical_search": "NOT RUN; requires separate explicit approval outside this notebook",
    "primary_test_evaluation": "NOT RUN",
}
print(json.dumps(report, indent=2))